# Page 0A · Transformers and Mixture of Experts

A light notebook, about ten minutes, with **no API key and no model download**. It opens up the two ideas on page 0A of the textbook using a few lines of numpy: how a transformer's attention lets each token look at the tokens before it, and how a mixture-of-experts router sends each token to only a few experts.

One honest caveat first: a real model *learns* its weights from trillions of tokens. Here the weights are random numbers, so the patterns you see are **not meaningful**. What is real is the arithmetic: the same steps, in the same order, that a trained transformer performs.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(7)             # fixed seed: everyone gets the same numbers
tokens = ["Deere", "grew", "revenue", "faster", "than", "Caterpillar"]
n, d = len(tokens), 8                      # 6 tokens, each described by 8 numbers
print(n, "tokens,", d, "numbers per token")

## 1. Every token becomes a list of numbers

The model cannot read words, only numbers. Each token is looked up in a table and becomes a vector (here 8 numbers; a real model uses thousands). This table is the **embedding**.

In [ ]:
embedding_table = {t: rng.normal(size=d) for t in tokens}
X = np.stack([embedding_table[t] for t in tokens])      # shape (6, 8): one row per token
pd.DataFrame(X.round(2), index=tokens).head(3)

## 2. Attention: each token asks a question of every earlier token

Each token is turned into three vectors: a **query** (what am I looking for?), a **key** (what do I offer?) and a **value** (what do I pass on?). A token's query is compared with every key. The higher the match, the more of that token's value it takes in.

The last step hides the future: when predicting the next word, a token may look only at itself and the tokens **before** it.

In [ ]:
Wq, Wk, Wv = (rng.normal(size=(d, d)) for _ in range(3))   # three learned matrices, random here
Q, K, V = X @ Wq, X @ Wk, X @ Wv

scores = Q @ K.T / np.sqrt(d)                              # how well does each query match each key?
mask = np.triu(np.ones((n, n), dtype=bool), k=1)           # True above the diagonal = the future
scores[mask] = -np.inf                                     # a token cannot see later tokens

weights = np.exp(scores - scores.max(axis=1, keepdims=True))
weights /= weights.sum(axis=1, keepdims=True)              # softmax: each row now adds up to 1

print("Each row adds up to:", weights.sum(axis=1).round(2))
pd.DataFrame(weights.round(2), index=tokens, columns=tokens)

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
ax.imshow(weights, cmap="Blues")
ax.set_xticks(range(n)); ax.set_xticklabels(tokens, rotation=45, ha="right")
ax.set_yticks(range(n)); ax.set_yticklabels(tokens)
ax.set_xlabel("looked at"); ax.set_ylabel("reading")
ax.set_title("Attention weights (random model)")
plt.tight_layout(); plt.show()

Read a row as "when this token is being read, how much attention does it pay to each token?" The empty upper triangle is the hidden future. The output for each token is the weighted mix of the values:

In [ ]:
output = weights @ V                       # shape (6, 8): a new, context-aware vector for each token
print("Same shape in and out:", X.shape, "->", output.shape)
print("'Caterpillar' now carries a mix of all six tokens' values, not just its own.")

## 3. Why long inputs cost more

Every token looks at every earlier token, so the weights table has about n × n / 2 cells. Double the text and the table roughly **quadruples**. This is one reason the context window has a limit.

In [ ]:
rows = [{"tokens in the text": length, "cells in the attention table": length * (length + 1) // 2}
        for length in (100, 200, 400, 800)]
df = pd.DataFrame(rows)
df["compared with 100 tokens"] = (df["cells in the attention table"] / df["cells in the attention table"][0]).round(1)
df

## 4. Mixture of experts: a router picks a few experts per token

In a **dense** model every token passes through the whole layer. In a **mixture-of-experts** model the layer is split into many small blocks, the experts, and a small **router** scores them for each token and sends it to only the top few.

In [ ]:
n_experts, top_k = 8, 2
router = rng.normal(size=(d, n_experts))                    # the router: one score per expert, per token

router_scores = output @ router                             # use the attention output from step 2
chosen = np.argsort(-router_scores, axis=1)[:, :top_k]      # the top 2 experts for each token

table = pd.DataFrame({"token": tokens,
                      "experts used": [sorted(int(e) for e in row) for row in chosen]})
table

In [ ]:
used = top_k / n_experts
print(f"Each token uses {top_k} of {n_experts} experts, so {used:.0%} of the layer does work per token.")
print(f"A dense layer of the same total size would use 100%.")

heat = np.zeros((n, n_experts))
for i, row in enumerate(chosen):
    heat[i, row] = 1
fig, ax = plt.subplots(figsize=(6, 3.2))
ax.imshow(heat, cmap="Oranges", aspect="auto")
ax.set_yticks(range(n)); ax.set_yticklabels(tokens)
ax.set_xticks(range(n_experts)); ax.set_xticklabels([f"E{i}" for i in range(n_experts)])
ax.set_title("Which experts each token is sent to")
plt.tight_layout(); plt.show()

The model still *holds* all eight experts, so it knows as much as a large network. It only *runs* two per token, so it costs about as much as a much smaller one. That is the trade DeepSeek-V3 makes: 671 billion weights held, about 37 billion used per token.

The experts are not neat specialists such as "finance" or "grammar". The router learns its own way of dividing the work.

## Try it

1. Change `top_k` to 1, then 4. How does the share of the layer used per token change?
2. In step 2, remove the two lines that hide the future (`mask` and `scores[mask] = -np.inf`). What does the weights table look like now, and why can a model that predicts the next word not do this?
3. In step 3, add a length of 1,600 tokens. By how much does the table grow compared with 800?